# ALS — Alternating Least Squares

## Learning Objectives

1. **Formulate** matrix factorization as a regularised least-squares objective
2. **Derive** the closed-form ALS update for user and item factor vectors
3. **Explain** why alternating least squares converges and is parallelisable
4. **Describe** the implicit feedback extension (Hu-Koren-Volinsky 2008)
5. **Implement** ALS for both explicit and implicit ratings


## Problem Statement

### Matrix Factorization for Recommendation

Given a ratings matrix $R \in \mathbb{R}^{m \times n}$ (users × items, mostly missing), we want to find:
$$R \approx U V^\top$$
where $U \in \mathbb{R}^{m \times k}$ (user factors) and $V \in \mathbb{R}^{n \times k}$ (item factors), with $k \ll \min(m, n)$.

**Objective (explicit feedback):**
$$\min_{U,V} \sum_{(u,i) \text{ observed}} (r_{ui} - u_u^\top v_i)^2 + \lambda \left(\sum_u \|u_u\|^2 + \sum_i \|v_i\|^2\right)$$

This is non-convex in $(U, V)$ jointly, but **convex in $U$ given $V$**, and convex in $V$ given $U$.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="370" font-family="monospace" font-size="12">
  <rect width="820" height="370" fill="#fafafa" rx="8"/>
  <defs><marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#999"/></marker></defs>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">ALS — Alternating Least Squares Matrix Factorization</text>

  <!-- Matrix decomposition diagram -->
  <text x="170" y="52" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">R ≈ U · Vᵀ</text>
  <!-- R matrix -->
  <rect x="20" y="60" width="90" height="100" rx="3" fill="white" stroke="#4e79a7" stroke-width="2"/>
  <text x="65" y="116" text-anchor="middle" fill="#4e79a7" font-size="14" font-weight="bold">R</text>
  <text x="65" y="134" text-anchor="middle" fill="#555" font-size="10">m × n</text>
  <!-- "=" -->
  <text x="125" y="116" text-anchor="middle" fill="#333" font-size="16">=</text>
  <!-- U matrix -->
  <rect x="140" y="60" width="50" height="100" rx="3" fill="#dbeeff" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="165" y="116" text-anchor="middle" fill="#4e79a7" font-size="14" font-weight="bold">U</text>
  <text x="165" y="134" text-anchor="middle" fill="#555" font-size="10">m×k</text>
  <!-- "·" -->
  <text x="205" y="116" text-anchor="middle" fill="#333" font-size="16">·</text>
  <!-- V^T matrix -->
  <rect x="218" y="60" width="100" height="40" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="268" y="84" text-anchor="middle" fill="#f28e2b" font-size="14" font-weight="bold">Vᵀ</text>
  <text x="268" y="134" text-anchor="middle" fill="#555" font-size="10">k×n</text>

  <!-- ALS update equations -->
  <rect x="350" y="42" width="450" height="140" rx="5" fill="#f5f5f5" stroke="#ccc"/>
  <text x="575" y="62" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">ALS Update Rules (closed-form least squares)</text>
  <text x="570" y="86" text-anchor="middle" fill="#333" font-size="11">Fix V, solve for each row uᵤ of U:</text>
  <text x="570" y="106" text-anchor="middle" fill="#333" font-size="12">uᵤ = (VᵤᵀVᵤ + λI)⁻¹ Vᵤᵀ rᵤ</text>
  <text x="570" y="124" text-anchor="middle" fill="#555" font-size="10">Vᵤ = rows of V for items rated by user u; rᵤ = user u's ratings</text>
  <text x="570" y="146" text-anchor="middle" fill="#333" font-size="11">Fix U, solve for each row vᵢ of V:</text>
  <text x="570" y="166" text-anchor="middle" fill="#333" font-size="12">vᵢ = (UᵢᵀUᵢ + λI)⁻¹ Uᵢᵀ rᵢ</text>

  <!-- Alternating arrows -->
  <rect x="20" y="175" width="300" height="80" rx="5" fill="#e8f0fb" stroke="#4e79a7"/>
  <text x="170" y="198" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Even iterations</text>
  <text x="170" y="216" text-anchor="middle" fill="#333" font-size="11">Fix V → Update all rows of U</text>
  <text x="170" y="234" text-anchor="middle" fill="#555" font-size="10">(each row is independent → parallel)</text>
  <text x="170" y="248" text-anchor="middle" fill="#555" font-size="10">O(|R|k + mk²) per step</text>

  <rect x="340" y="175" width="300" height="80" rx="5" fill="#fff4e0" stroke="#f28e2b"/>
  <text x="490" y="198" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">Odd iterations</text>
  <text x="490" y="216" text-anchor="middle" fill="#333" font-size="11">Fix U → Update all rows of V</text>
  <text x="490" y="234" text-anchor="middle" fill="#555" font-size="10">(each row is independent → parallel)</text>
  <text x="490" y="248" text-anchor="middle" fill="#555" font-size="10">O(|R|k + nk²) per step</text>

  <line x1="320" y1="215" x2="338" y2="215" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="340" y1="222" x2="322" y2="222" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Implicit feedback box -->
  <rect x="20" y="270" width="780" height="88" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="290" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Implicit Feedback ALS (Hu, Koren, Volinsky 2008)</text>
  <text x="410" y="312" text-anchor="middle" fill="#333" font-size="11">Binary confidence: cᵤᵢ = 1 + α·rᵤᵢ (plays/clicks/views),  pᵤᵢ = 𝟙[rᵤᵢ > 0]</text>
  <text x="410" y="330" text-anchor="middle" fill="#333" font-size="11">Objective: Σᵤᵢ cᵤᵢ(pᵤᵢ − uᵤᵀvᵢ)² + λ(Σᵤ‖uᵤ‖² + Σᵢ‖vᵢ‖²)</text>
  <text x="410" y="348" text-anchor="middle" fill="#555" font-size="10">High confidence on observed interactions; low confidence (c=1) on unobserved (assumed preference=0)</text>
</svg>
'''
display(SVG(svg))


## Derivation

### ALS for Explicit Feedback

**Fix $V$, solve for row $u_u$ of $U$:**

Let $V_u$ = rows of $V$ for items rated by user $u$, and $r_u$ = user $u$'s observed ratings. The subproblem is:
$$u_u = \arg\min_{x} \|V_u x - r_u\|^2 + \lambda \|x\|^2$$

This is ridge regression with closed-form solution:
$$u_u = (V_u^\top V_u + \lambda I)^{-1} V_u^\top r_u$$

**Fix $U$, solve for $v_i$ symmetrically.**

Each row update is independent → **embarrassingly parallel**.

### ALS for Implicit Feedback (HKV 2008)

Observations are interaction counts (plays, clicks, views) rather than explicit ratings. Define:
- **Confidence:** $c_{ui} = 1 + \alpha r_{ui}$ where $r_{ui}$ is the raw count
- **Preference:** $p_{ui} = \mathbb{1}[r_{ui} > 0]$

**Objective:**
$$\min_{U,V} \sum_{u,i} c_{ui}(p_{ui} - u_u^\top v_i)^2 + \lambda(\|U\|_F^2 + \|V\|_F^2)$$

**ALS update for $u_u$:**
$$u_u = (V^\top C^u V + \lambda I)^{-1} V^\top C^u p_u$$
where $C^u = \text{diag}(c_{u1}, \ldots, c_{un})$.

This sums over **all items** (not just observed), making each update $O(nk^2)$. With the identity trick $V^\top C^u V = V^\top V + V^\top (C^u - I) V$, reduce to $O(|\mathcal{R}_u| k^2 + k^3)$.

### Convergence

ALS is not guaranteed to find the global optimum (non-convex overall), but in practice converges to good local optima within 10–30 iterations. RMSE monotonically decreases within each half-step.


## Algorithm Steps

1. **Initialise** $V$ randomly (small values); set regularisation $\lambda$, rank $k$
2. **Repeat** for $T$ iterations:
   - For each user $u$: solve $u_u = (V_u^\top V_u + \lambda I)^{-1} V_u^\top r_u$
   - For each item $i$: solve $v_i = (U_i^\top U_i + \lambda I)^{-1} U_i^\top r_i$
3. **Output** $U, V$; predict $\hat{r}_{ui} = u_u^\top v_i$


In [ ]:
import numpy as np


def als(R, k=10, n_iter=20, lambda_reg=0.1, seed=42):
    """
    Alternating Least Squares matrix factorization.

    Inputs
    ------
    R         : np.ndarray shape (m, n), with np.nan for missing entries
    k         : int — latent factor dimension
    n_iter    : int — number of ALS iterations (each = one U update + one V update)
    lambda_reg: float λ — L2 regularisation

    Returns
    -------
    U : np.ndarray (m, k)
    V : np.ndarray (n, k)
    """
    rng = np.random.default_rng(seed)
    m, n = R.shape

    # Initialise V randomly; U will be solved on first iteration
    V = rng.normal(0, 0.1, (n, k))
    U = np.zeros((m, k))

    # Mask of observed entries
    observed = ~np.isnan(R)

    def solve_row(vec, factor_matrix, obs_mask):
        """Solve one ALS subproblem: (AᵀA + λI)x = Aᵀb"""
        A = factor_matrix[obs_mask]          # shape (n_obs, k)
        b = vec[obs_mask]                    # shape (n_obs,)
        ATA = A.T @ A + lambda_reg * np.eye(k)
        ATb = A.T @ b
        return np.linalg.solve(ATA, ATb)

    rmse_history = []

    for iteration in range(n_iter):
        # ── Update U: fix V ────────────────────────────────────────────────────
        for u in range(m):
            obs_u = observed[u]                # which items user u rated
            if obs_u.sum() == 0:
                continue
            U[u] = solve_row(R[u], V, obs_u)

        # ── Update V: fix U ────────────────────────────────────────────────────
        for i in range(n):
            obs_i = observed[:, i]             # which users rated item i
            if obs_i.sum() == 0:
                continue
            V[i] = solve_row(R[:, i], U, obs_i)

        # ── RMSE on observed entries ────────────────────────────────────────────
        R_hat = U @ V.T
        diff = R_hat[observed] - R[observed]
        rmse = np.sqrt(np.mean(diff**2))
        rmse_history.append(rmse)

    return U, V, rmse_history


def als_implicit(C, k=10, n_iter=15, lambda_reg=0.1, seed=42):
    """
    ALS for implicit feedback (Hu-Koren-Volinsky 2008).

    C : np.ndarray (m, n) confidence matrix; C[u,i] = 1 + alpha * r[u,i]
    P : binary preference matrix; P[u,i] = 1 if C[u,i] > 1 else 0
    """
    rng = np.random.default_rng(seed)
    m, n = C.shape
    P = (C > 1).astype(float)

    U = rng.normal(0, 0.1, (m, k))
    V = rng.normal(0, 0.1, (n, k))
    I_k = np.eye(k)

    for _ in range(n_iter):
        VTV = V.T @ V
        for u in range(m):
            c_u = np.diag(C[u])              # (n, n) diagonal confidence
            p_u = P[u]                        # (n,) preference
            A = V.T @ c_u @ V + lambda_reg * I_k
            b = V.T @ c_u @ p_u
            U[u] = np.linalg.solve(A, b)

        UTU = U.T @ U
        for i in range(n):
            c_i = np.diag(C[:, i])
            p_i = P[:, i]
            A = U.T @ c_i @ U + lambda_reg * I_k
            b = U.T @ c_i @ p_i
            V[i] = np.linalg.solve(A, b)

    return U, V


# ── Demo ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
m, n, k_true = 30, 20, 3
U_true = rng.normal(0, 1, (m, k_true))
V_true = rng.normal(0, 1, (n, k_true))
R_full = U_true @ V_true.T + rng.normal(0, 0.2, (m, n))

# Mask 40% of entries as missing
R_obs = R_full.copy()
mask = rng.random((m, n)) < 0.4
R_obs[mask] = np.nan

U_hat, V_hat, rmse_hist = als(R_obs, k=3, n_iter=30, lambda_reg=0.05)
R_hat = U_hat @ V_hat.T

test_rmse = np.sqrt(np.mean((R_hat[mask] - R_full[mask])**2))
print(f"ALS RMSE on held-out entries: {test_rmse:.4f}")
print(f"Training RMSE trajectory (first/mid/last): "
      f"{rmse_hist[0]:.4f} → {rmse_hist[14]:.4f} → {rmse_hist[-1]:.4f}")
